# Phase 2 — QLoRA Fine-Tuning: Llama-3-8B-Instruct for Civic Complaint Routing

**Runtime required:** Colab Free Tier with a **T4 GPU**.
Go to `Runtime > Change runtime type > T4 GPU` before running anything below.

This notebook:
1. Installs Unsloth + dependencies
2. Loads `unsloth/llama-3-8b-Instruct-bnb-4bit` in 4-bit precision
3. Loads your `train.jsonl` (generated in Phase 1) and formats it for training
4. Attaches LoRA adapters and fine-tunes with **target-only loss masking**
   (loss is computed only on the assistant's JSON output, not the user's input)
5. Runs a quick before/after inference sanity check
6. Pushes the trained LoRA adapter to the Hugging Face Hub for Phase 3

**Before you start:** generate a free Hugging Face **write** access token at
https://huggingface.co/settings/tokens — you'll paste it in Cell 2.


## 1. Install dependencies

This takes ~2-3 minutes on first run.

In [ ]:
%%capture
import os

# Unsloth ships a one-liner that auto-detects the Colab GPU and installs
# the correct compatible versions of torch/xformers/bitsandbytes for you.
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- check Runtime > Change runtime type > T4 GPU")


CUDA available: True
GPU: Tesla T4


## 2. Hugging Face authentication

Needed to push the trained adapter to the Hub at the end of this notebook.
Get a **write**-scoped token from https://huggingface.co/settings/tokens


In [ ]:
from huggingface_hub import login
from getpass import getpass

from google.colab import userdata
x = userdata.get('HF_TOKEN')

login(token=x)


## 3. Load the base model in 4-bit (QLoRA)

`unsloth/llama-3-8b-Instruct-bnb-4bit` is Unsloth's pre-quantized version of
Meta's Llama-3-8B-Instruct -- free to use, no gated-repo approval needed,
and loads ~4x faster than quantizing the full-precision model yourself.

Loading in 4-bit precision is what makes this fit on a free T4 GPU
(~16GB VRAM) -- the full-precision model would need ~32GB+.


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # complaints + JSON labels are short; this is generous headroom
dtype = None             # auto-detect: bfloat16 if supported, else float16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model loaded successfully.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded successfully.


## 4. Attach LoRA adapters

Instead of fine-tuning all 8 billion parameters, we freeze the base model
and only train small low-rank "adapter" matrices injected into specific layers.
This is what makes QLoRA fine-tuning feasible on a single free GPU --
typically well under 1% of the total parameters actually get updated.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank -- 16 is a solid default for this size of task
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,            # 0 is optimized/recommended by Unsloth for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # reduces VRAM usage further
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()


Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## 5. Load and format the Phase 1 dataset

Upload your `train.jsonl` (from `data/processed/train.jsonl` in Phase 1) to this
Colab session using the file browser on the left, or mount Google Drive --
either works, just adjust the path below.

Each line is already in ChatML format:
```json
{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
```

We convert this into Llama-3's actual chat template string, which is what
the model needs to see during training.


In [ ]:
from datasets import load_dataset

# Adjust this path if you uploaded the file elsewhere or mounted Drive
DATASET_PATH = "train.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(dataset)} examples")
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 1200 examples
{'messages': [{'role': 'system', 'content': "You are a civic complaint structuring engine. Convert the user's message into a single valid JSON object matching the schema. Output ONLY the JSON object -- no explanations, no markdown."}, {'role': 'user', 'content': "It's BEEN DAYS since this area near LAKE MARKET is completely UNDER WATER! No one is coming to FIX this MESS and we are suffering! The roads are like PANCHAYAT, we can’t even step outside! It’s so frustrating! PLEASE do something about it NOW!"}, {'role': 'assistant', 'content': '{"category": "WATER_LOGGING", "location_raw": "Lake Market, Kolkata", "priority": "HIGH", "sentiment": "ANGRY", "description_summary": "Severe water logging issue in Lake Market area.", "language_detected": "EN", "requires_immediate_attention": true}'}]}


In [ ]:
def formatting_func(examples):
    """Applies Llama-3's chat template to each ChatML message list."""
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}


dataset = dataset.map(formatting_func, batched=True)
print(dataset[0]["text"][:9])  # sanity check: should show the formatted chat


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

<|begin_o


## 6. Train with target-only loss masking

This is the key efficiency detail from your project spec: **loss is computed
only on the assistant's JSON output tokens, not on the system prompt or the
user's raw complaint text.** This makes training converge faster and avoids
wasting gradient updates on text the model doesn't need to learn to generate.

Unsloth's `train_on_responses_only` utility handles this masking automatically
by matching the instruction/response template markers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,           # adjust based on dataset size; 3 is a safe default for ~1500 rows
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Mask the loss so it's computed ONLY on the assistant's JSON response,
# not on the system prompt or the user's raw complaint text.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

In [ ]:
# Quick VRAM check before the long run
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = torch.cuda.max_memory_reserved() / 1024**3
print(f"GPU = {gpu_stats.name}, max memory = {gpu_stats.total_memory / 1024**3:.1f} GB")
print(f"Memory reserved before training: {start_mem:.2f} GB")


GPU = Tesla T4, max memory = 14.6 GB
Memory reserved before training: 5.50 GB


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,200 | Num Epochs = 3 | Total steps = 450
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.522748
20,0.352182
30,0.266641
40,0.231984
50,0.225060
60,0.215945
70,0.202419
80,0.195283
90,0.191830
100,0.195734


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-450/tokenizer_config.json.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

## 7. Sanity-check inference

Before pushing anything to the Hub, run a couple of held-out-style prompts
through the fine-tuned model and eyeball whether the output is clean JSON
matching your schema.


In [ ]:
FastLanguageModel.for_inference(model)  # enables Unsloth's faster inference path

test_complaints = [
    "Bhai master canteen road pe heavy water logging hai, bikes are slipping",
    "There has been no streetlight on MG Road for a week, very unsafe at night",
]

for complaint in test_complaints:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a civic complaint structuring engine. Convert the user's "
                "message into a single valid JSON object matching the schema. "
                "Output ONLY the JSON object -- no explanations, no markdown."
            ),
        },
        {"role": "user", "content": complaint},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=200, use_cache=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    print(f"INPUT:  {complaint}")
    print(f"OUTPUT: {response}")
    print("-" * 80)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_m

INPUT:  Bhai master canteen road pe heavy water logging hai, bikes are slipping
OUTPUT: {"category": "WATER_LOGGING", "location_raw": "Master Canteen Road, Kolkata", "priority": "MEDIUM", "sentiment": "NEUTRAL", "description_summary": "Heavy water logging on Master Canteen Road causing slippery conditions for vehicles.", "language_detected": "EN", "requires_immediate_attention": false}
--------------------------------------------------------------------------------
INPUT:  There has been no streetlight on MG Road for a week, very unsafe at night
OUTPUT: {"category": "STREETLIGHT", "location_raw": "MG Road", "priority": "MEDIUM", "sentiment": "NEUTRAL", "description_summary": "Streetlight outage on MG Road", "language_detected": "EN", "requires_immediate_attention": false}
--------------------------------------------------------------------------------


In [ ]:
import os

# Remove any cached token files that might be silently overriding your fresh login
for path in [
    os.path.expanduser("~/.cache/huggingface/token"),
    os.path.expanduser("~/.huggingface/token"),
]:
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed cached token at {path}")

from huggingface_hub import login
hf_token = ""  # the one whoami() just confirmed as write
login(token=hf_token, add_to_git_credential=True)

from huggingface_hub import whoami
print(whoami())

Removed cached token at /root/.cache/huggingface/token
{'type': 'user', 'id': '68b6e4ab79d03eb8e3f50980', 'name': 'Gourabswain', 'fullname': 'GOURAB SWAIN', 'email': 'gourabswain526@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/4275ddf9fa14719b3c385f2322ae317a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'CUSTOMAPI', 'role': 'write', 'createdAt': '2026-06-25T04:41:36.053Z'}}}


## 8. Push the LoRA adapter to Hugging Face Hub

This uploads ONLY the small LoRA adapter weights (a few hundred MB at most),
not the full 8B base model -- Phase 3 will load the base model fresh from
Unsloth/Hugging Face and attach this adapter on top of it.

**Replace `your-username/civic-complaint-router-lora` with your actual HF username.**


In [ ]:
# from huggingface_hub import login

# # Force a fresh login, overwriting any cached/stale credential
# login(token="", add_to_git_credential=True)

In [ ]:
GGUF_REPO_ID = "Gourabswain/civic-complaint-router-gguf"  # note: lowercase 'w' per your actual namespace "Gourabswain"

model.push_to_hub_gguf(
    GGUF_REPO_ID,
    tokenizer,
    quantization_method="q4_k_m",
    token=hf_token,
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_hp8jm0a8/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3-8b-instruct.Q4_K_M.gguf:   0%|          | 14.3MB / 4.92GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Gourabswain/civic-complaint-router-gguf
Unsloth: Cleaning up temporary files...


'Gourabswain/civic-complaint-router-gguf'

In [ ]:
HF_REPO_ID = "GourabSwain/civic-complaint-router-lora"  # <-- CHANGE THIS

model.push_to_hub(HF_REPO_ID, token=hf_token)
tokenizer.push_to_hub(HF_REPO_ID, token=hf_token)

print(f"Adapter pushed to: https://huggingface.co/{HF_REPO_ID}")


HfHubHTTPError: (Request ID: Root=1-6a3cbccf-68fabbae3ab966f61480d6b2;cca9989d-ff11-44a2-9559-f713a8c26755)

403 Forbidden: You don't have the rights to create a model under the namespace "GourabSwain".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
from huggingface_hub import whoami
info = whoami()
print(info)

{'type': 'user', 'id': '68b6e4ab79d03eb8e3f50980', 'name': 'Gourabswain', 'fullname': 'GOURAB SWAIN', 'email': 'gourabswain526@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/4275ddf9fa14719b3c385f2322ae317a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'CUSTOMAPI', 'role': 'write', 'createdAt': '2026-06-25T04:41:36.053Z'}}}


## Done — Phase 2 complete

Your LoRA adapter is now on the Hugging Face Hub. In Phase 3, `src/model_loader.py`
will load the base `unsloth/llama-3-8b-Instruct-bnb-4bit` model and attach this
adapter for production inference inside the LangChain + Pydantic validation pipeline.

**Note the repo ID you used above** -- you'll need it for Phase 3.
